# Lectures 20–21: Vector Operations

## Dissolve, Clip and Buffer

In the previous lectures, we used **spatial joins** to attach information from one spatial layer to another.

In these lectures, we move to a different set of operations. Instead of only asking *which features are related spatially?*, we will start **changing the geometry itself**.

We will work with three important vector operations:

- **Dissolve** — combine neighbouring or related geometries.
- **Clip** — cut one spatial layer using the boundary of another layer.
- **Buffer** — create a zone at a specified distance around a spatial feature.

The examples use Indian pincode boundaries, railway lines, Delhi wards, and Delhi POIs.

## Data used in these lectures

The notebook assumes that the files are stored in:

`C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data`

The files used are:

- `All_India_pincode_Boundary-19312_shp.shp`
- `ind_railways_lines.geojson`
- `delhi_wards.geojson`
- `delhi_pois.geojson`

> **Important:** Change `data_path` according to where the data are stored on your own computer.

If the pincode layer is stored as a shapefile, remember that a shapefile is made up of several files (`.shp`, `.shx`, `.dbf`, `.prj`, etc.). Keep all of them together in the same folder.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

data_path = r"C:\Teaching\IIT Tirupati\UISA\2026\uisa_codes\data"

# Lecture 20 — Dissolve and Clip

We begin with **dissolve**.

A dissolve operation combines geometries based on a common attribute.

Suppose a dataset contains many small polygons:

```text
Pincode 1 ┐
Pincode 2 ├── Andhra Pradesh Postal Circle
Pincode 3 ┘

Pincode 4 ┐
Pincode 5 ├── Tamil Nadu Postal Circle
Pincode 6 ┘
```

If we dissolve using the postal-circle field, all pincode polygons belonging to the same circle are merged.

The important idea is:

> **Dissolve reduces the number of spatial features by merging geometries that belong to the same group.**

## Loading the pincode layer

Each row in the pincode dataset represents a pincode area.

Before performing any operation, inspect:

- the number of rows;
- the CRS;
- the available columns;
- the geometry type.

This is always a good first step when opening a new spatial dataset.

In [ ]:
pincodes = gpd.read_file(
    data_path + r"\India_Pincodes\All_India_pincode_Boundary-19312_shp.shp"
)

print("Number of features:", len(pincodes))
print("CRS:", pincodes.crs)
print("Geometry types:")
print(pincodes.geom_type.value_counts())

display(pincodes.head())

The fields `Circle` and `Division` are especially useful in this dataset.

A **postal circle** is a much larger geographic unit than an individual pincode area. Therefore, many pincode polygons can belong to the same `Circle`.

In [ ]:
pincodes["Circle"].value_counts()

## Looking at the original pincode polygons

Before dissolving the data, it is useful to see the original level of detail.

Each boundary in the map below represents an individual pincode polygon.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

pincodes.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.2
)

ax.set_title("Original Pincode Boundaries")
ax.set_axis_off()

plt.show()

# Dissolving by an attribute

The basic GeoPandas syntax is:

`GeoDataFrame.dissolve(by="column_name")`

Here we use:

`by="Circle"`

GeoPandas will:

1. look at the value of `Circle` for every pincode;
2. group pincodes having the same value;
3. merge their polygon geometries;
4. return one geometry for each postal circle.

This is similar to `groupby()` in Pandas, but there is an important difference:

> **`groupby()` groups rows and summarizes attributes. `dissolve()` also combines the geometries.**

In [ ]:
circles = pincodes.dissolve(by="Circle").reset_index()

print("Original pincode features:", len(pincodes))
print("Features after dissolve:", len(circles))

display(circles.head())

### Why do we use `.reset_index()`?

After `dissolve(by="Circle")`, the `Circle` field becomes the index of the result.

Using:

`reset_index()`

moves it back into a normal DataFrame column, which is often easier to inspect and use later.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

circles.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=0.7
)

ax.set_title("Pincode Areas Dissolved by Postal Circle")
ax.set_axis_off()

plt.show()

## Dissolve without `by=`

Sometimes we do not want to preserve any internal grouping at all.

For example, the Delhi ward layer contains many ward polygons. If our purpose is only to obtain the **outer boundary of India**, we can dissolve all wards into one geometry.

When `dissolve()` is used without a `by=` argument:

> **All geometries are combined into a single feature.**

This is particularly useful when we need a single polygon to act as a **mask** for another spatial operation.

In [ ]:
India = circles.dissolve()

print("Number of ward features:", len(circles))
print("Features after dissolve:", len(India))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

India.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1.2
)

ax.set_title("India After Dissolving the Ward Boundaries")
ax.set_axis_off()

plt.show()

The internal ward boundaries have disappeared.

We now have a single geometry representing the overall India boundary.

This leads naturally to the next vector operation: **clip**.

# Clipping

A clip operation works like a **cookie cutter**.

We have:

- an **input layer** that we want to cut; and
- a **mask layer** that defines the area we want to keep.

For example:

```text
All India railway lines
          +
Delhi boundary
          ↓
Railway lines inside Delhi only
```

The basic syntax is:

`gpd.clip(input_layer, mask_layer)`

Only the parts of the input geometries that fall inside the mask are retained.

## Loading the railway layer

The railway data contain line geometries covering a much larger area than Delhi.

Our goal is to keep only the portions of railway lines that lie within the Delhi boundary.

In [ ]:
rails = gpd.read_file(
    data_path + r"\ind_railways_lines.geojson"
)

print("Railway features:", len(rails))
print("Railway CRS:", rails.crs)
print("Delhi CRS:", delhi.crs)

display(rails.head())

In [ ]:
circles.head()

In [ ]:
## Selecting Delhi Circle from circle data 

delhi = circles[circles["Circle"]=="Delhi"]

fig, ax = plt.subplots(figsize=(8, 8))

delhi.plot(
    ax=ax,
    facecolor="none",
    edgecolor="black",
    linewidth=1.2
)

ax.set_title("Delhi circle from Indian pincodes data")
ax.set_axis_off()

plt.show()

## CRS must match before clipping

Spatial operations compare coordinates between layers.

Therefore, the railway layer and the Delhi boundary must use the same CRS.

If the CRSs are different, we reproject the railway layer to match the Delhi boundary.

In [ ]:
if rails.crs != delhi.crs:
    rails = rails.to_crs(delhi.crs)

print("Railway CRS:", rails.crs)
print("Delhi CRS:", delhi.crs)

Before clipping, we can look at the railway lines around Delhi.

The railway layer still contains the complete railway geometries. The Delhi boundary simply shows the area that we are interested in.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

rails.plot(
    ax=ax,
    color="grey",
    linewidth=0.6,
    zorder=1
)

delhi.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.5,
    zorder=2
)

#xmin, ymin, xmax, ymax = delhi.total_bounds

#ax.set_xlim(xmin - 0.15, xmax + 0.15)
#ax.set_ylim(ymin - 0.15, ymax + 0.15)

ax.set_title("Railway Lines Around Delhi Before Clipping")
ax.set_axis_off()

plt.show()

## Performing the clip

Now we use the dissolved Delhi boundary as the clipping mask.

`rails` is the layer being cut.

`delhi` is the polygon that tells GeoPandas what area should be retained.

In [ ]:
rails_delhi = gpd.clip(rails, delhi)

print("Railway features before clip:", len(rails))
print("Railway features after clip:", len(rails_delhi))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

delhi.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="black",
    linewidth=1,
    zorder=1
)

rails_delhi.plot(
    ax=ax,
    color="red",
    linewidth=1.2,
    zorder=2
)

ax.set_title("Railway Lines Clipped to Delhi")
ax.set_axis_off()

plt.show()

## Spatial join versus clip

It is important not to confuse these two operations.

### Spatial join

A spatial join answers questions such as:

> Which ward contains this crash?

It mainly **adds attributes** from one layer to another.

The crash point itself is not cut or reshaped.

### Clip

A clip answers questions such as:

> Which parts of these railway lines fall inside Delhi?

It **changes the geometry** by removing portions outside the clipping boundary.

So:

> **Spatial join = attach information based on location.**

> **Clip = physically retain only the geometry inside a boundary.**

# Class Exercise 17 — Dissolve and Clip

Using the pincode and railway datasets:

1. Select only pincode polygons where `Division` is **Tirupati**.
2. Plot the original pincode polygons belonging to Tirupati Division.
3. Dissolve these polygons to create a single Tirupati Division boundary.
4. Make sure the railway layer and the dissolved Tirupati boundary use the same CRS.
5. Clip the railway lines using the dissolved Tirupati boundary.
6. Create a map showing:
   - the Tirupati Division boundary; and
   - the railway lines that remain after clipping.
7. Give the map an appropriate title and turn the axes off.

Think about the sequence:

**filter → dissolve → match CRS → clip → map**

In [ ]:
# Write your solution here

# Lecture 21 — Buffer

A **buffer** creates a zone around a spatial feature at a specified distance.

Examples include:

- 500 m around a hospital;
- 1 km around a railway station;
- 100 m on either side of a road;
- 2 km around a lake.

The geometry of the buffer depends on the original feature:

- **point → circular area**
- **line → corridor**
- **polygon → expanded area**

A buffer is therefore useful whenever our question involves **proximity or distance**.

## Buffer distance and CRS

This is one of the most important ideas in buffering.

Consider:

`geometry.buffer(500)`

What does `500` mean?

GeoPandas interprets the number using the **units of the CRS**.

If the layer is in a geographic CRS such as EPSG:4326, the coordinate units are **degrees**. A buffer of `500` would therefore not mean 500 metres.

For a distance-based operation, we should first use an appropriate **projected CRS** whose units are metres.

For Delhi we will use:

**EPSG:32643 — WGS 84 / UTM Zone 43N**

Then:

`buffer(500)`

means approximately **500 metres**.

## Loading the Delhi POIs

The POI layer contains point locations across Delhi.

We will create buffers around selected health-related POIs.

In [ ]:
pois = gpd.read_file(
    data_path + r"\delhi_pois.geojson"
)

print("Number of POIs:", len(pois))
print("CRS:", pois.crs)
print("Columns:", pois.columns.tolist())

display(pois.head())

Before selecting a category, it is useful to inspect the values in `PoI_Type`.

This helps us understand how the POIs have been classified in the source data.

In [ ]:
pois["PoI_Type"].value_counts().head(20)

## Reprojecting before buffering

We now project both the POIs and the Delhi wards to EPSG:32643.

The coordinates will then be measured in metres, which makes a 500 m buffer meaningful.

In [ ]:
pois_utm = pois.to_crs(epsg=32643)
wards_utm = wards.to_crs(epsg=32643)

print("POI CRS:", pois_utm.crs)
print("Ward CRS:", wards_utm.crs)

## Selecting health-related POIs

The expression below looks for POI types containing words such as:

- hospital
- clinic
- health

The `|` symbol inside the text pattern means **OR**.

We use this only to obtain a manageable and meaningful subset for the buffer example.

In [ ]:
health = pois_utm[
    pois_utm["PoI_Type"].str.contains(
        "hospital|clinic|health",
        case=False,
        na=False
    )
].copy()

print("Health-related POIs:", len(health))

display(
    health[["PoI_Type", "geometry"]].head()
)

Let us first plot the selected POIs without any buffer.

At this stage they are still point geometries.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

wards_utm.boundary.plot(
    ax=ax,
    color="grey",
    linewidth=0.4,
    zorder=1
)

health.plot(
    ax=ax,
    color="red",
    markersize=12,
    zorder=2
)

ax.set_title("Selected Health-related POIs in Delhi")
ax.set_axis_off()

plt.show()

# Creating a 500 m buffer

We make a copy of the health POIs and replace each point geometry with a 500 m buffer.

The key operation is:

`geometry.buffer(500)`

Because the CRS units are metres, each resulting polygon represents all locations within approximately 500 metres of the original POI.

In [ ]:
health_buffer = health.copy()

health_buffer["geometry"] = health_buffer.geometry.buffer(500)

display(
    health_buffer[["PoI_Type", "geometry"]].head()
)

The geometry has now changed:

**Before:** Point

**After:** Polygon

The original health POI GeoDataFrame `health` is still available, while `health_buffer` contains the new buffer polygons.

In [ ]:
print("Original geometry:")
print(health.geom_type.value_counts())

print("\nBuffered geometry:")
print(health_buffer.geom_type.value_counts())

## Mapping the buffers

The ward boundaries provide geographic context.

The buffer polygons are drawn above them, while the original POI points are placed on top.

We already learnt that `zorder` controls the visual stacking order:

- lower `zorder` → lower layer;
- higher `zorder` → drawn on top.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

wards_utm.boundary.plot(
    ax=ax,
    color="grey",
    linewidth=0.4,
    zorder=1
)

health_buffer.plot(
    ax=ax,
    alpha=0.30,
    edgecolor="black",
    linewidth=0.3,
    zorder=2
)

health.plot(
    ax=ax,
    color="red",
    markersize=10,
    zorder=3
)

ax.set_title("500 m Buffers Around Health-related POIs")
ax.set_axis_off()

plt.show()

## Changing the buffer distance

The number inside `buffer()` controls the distance.

For example:

- `buffer(250)` → 250 m
- `buffer(500)` → 500 m
- `buffer(1000)` → 1 km

A larger buffer does not mean that the POI itself changes. It means that we are asking about a larger **zone of proximity** around it.

The appropriate distance should always come from the purpose of the analysis rather than being chosen arbitrarily.

## Overlapping buffers and dissolve

Individual POI buffers may overlap.

If we want to identify the **combined area covered by all 500 m buffers**, we can dissolve them.

This is a useful connection between the two vector operations learnt in these lectures:

**buffer → dissolve**

Before dissolving, overlapping parts belong to several individual buffer polygons.

After dissolving, the overlapping polygons are combined into a single coverage geometry.

In [ ]:
health_coverage = health_buffer.dissolve()

fig, ax = plt.subplots(figsize=(10, 9))

wards_utm.boundary.plot(
    ax=ax,
    color="grey",
    linewidth=0.4,
    zorder=1
)

health_coverage.plot(
    ax=ax,
    alpha=0.35,
    edgecolor="black",
    zorder=2
)

health.plot(
    ax=ax,
    color="red",
    markersize=8,
    zorder=3
)

ax.set_title("Combined 500 m Health POI Buffer Coverage")
ax.set_axis_off()

plt.show()

Because the layer is in a projected CRS measured in metres, we can also calculate the total combined buffer area.

There are 1,000,000 square metres in one square kilometre.

In [ ]:
health_coverage["area_km2"] = (
    health_coverage.geometry.area / 1_000_000
)

health_coverage[["area_km2"]]

# Class Exercise 18 — Buffering POIs

Using `delhi_pois.geojson`:

1. Select POIs whose `PoI_Type` contains **school**, **college**, or **university**.
2. Reproject the POIs to EPSG:32643 if needed.
3. Create a **500 m buffer** around each selected education POI.
4. Plot:
   - Delhi ward boundaries;
   - the 500 m buffer polygons; and
   - the original education POI points.
5. Use `zorder` so that the points remain visible above the buffer polygons.
6. Dissolve all the buffer polygons to create one combined coverage layer.
7. Calculate the total dissolved buffer area in square kilometres.

Think about why the dissolved coverage area will generally be **smaller than the sum of the areas of all individual buffers**.

In [ ]:
# Write your solution here

# Summary

In these two lectures we used three vector-processing operations.

### Dissolve

Combines geometries.

Examples:

- many pincode polygons → postal circles;
- many Delhi wards → one Delhi boundary.

### Clip

Keeps only the parts of one layer that fall inside another geometry.

Example:

- all railway lines → railway lines inside Delhi.

### Buffer

Creates a distance-based zone around a feature.

Example:

- health POI point → 500 m surrounding area.

The three operations answer different questions:

| Operation | Main question |
|---|---|
| Dissolve | Which geometries should be combined? |
| Clip | Which parts of this layer fall inside my study area? |
| Buffer | What area lies within a given distance of this feature? |

A crucial rule for distance and area calculations is:

> **Use a suitable projected CRS when the operation depends on metres, kilometres or square kilometres.**